# Подготовка данных для обучения моделей

__Автор задач: Блохин Н.В. (NVBlokhin@fa.ru)__

Материалы:
* https://scikit-learn.org/stable/modules/compose.html#pipeline-chaining-estimators
* https://pytorch.org/docs/stable/data.html
* https://pytorch.org/tutorials/beginner/data_loading_tutorial.html
* Deep Learning with PyTorch (2020) Авторы: Eli Stevens, Luca Antiga, Thomas Viehmann


## Задачи для совместного разбора

1. Создайте синтетический датасет для задачи регрессии и представьте его в виде `torch.utils.data.Dataset`

In [122]:
import pandas as pd
from torch.utils.data import Dataset

## Задачи для самостоятельного решения

<p class="task" id="1"></p>

1\. Считайте файл `bank-full.csv` ([источник](https://www.kaggle.com/datasets/hariharanpavan/bank-marketing-dataset-analysis-classification)) в виде `pd.DataFrame`.

Опишите класс `BankDatasetBase`. Решение должно удовлетворять следующим критериям:

* класс наследуется от `torch.utils.data.Dataset`;
* при создании объекта в конструктор передается набор данных в виде `pd.DataFrame`;
* объекты класса имеют поля `X` и `y` с признаками и метками соответственно;
* класс реализует интерфейс последовательностей (`__getitem__` + `__len__`);
* `obj[i]` возвращает кортеж, содержащий `i`-ую строку из `obj.X` (серию) и `i`-ую строку из `obj.y` (строку).
    
Создайте объект класса `BankDatasetBase` и продемонстрируйте работоспособность.

- [ ] Проверено на семинаре

In [123]:
class BankDatasetBase(Dataset):
    def __init__(self, data: pd.DataFrame) -> None:
      self.X = data.drop('y', axis=1) #features остаются, опуская значения target
      self.y = data['y'] #выбираются из данных только target

    def __getitem__(self, idx: int) -> tuple:
        return (self.X.iloc[idx], self.y.iloc[idx]) #iloc - переход на числовое индексирование

    def __len__(self) -> int:
        return len(self.X)

In [124]:
df = pd.read_csv('/content/bank-full.csv')

In [125]:
data = BankDatasetBase(df)

print("Длина датасета:", len(data)) #Кол-во строчек датасета

x, y = data[0] #Получение первого элемента
print("\nПервый пример:")
print(x)
print("Метка:", y)

x, y = data[777] #Получение 777-го элемента
print("\nЭлемент с индексом 10000:")
print(x)
print("Метка:", y)

print("\nТипы:") #Типы целевой функции и features
print("Тип X:", type(x))
print("Тип y:", type(y))

Длина датасета: 45211

Первый пример:
age                  58
job          management
marital         married
education      tertiary
default              no
balance            2143
housing             yes
loan                 no
contact         unknown
day                   5
month               may
duration            261
campaign              1
pdays                -1
previous              0
poutcome        unknown
Name: 0, dtype: object
Метка: no

Элемент с индексом 10000:
age                  45
job          management
marital         married
education      tertiary
default              no
balance             103
housing             yes
loan                 no
contact         unknown
day                   7
month               may
duration             35
campaign              4
pdays                -1
previous              0
poutcome        unknown
Name: 777, dtype: object
Метка: no

Типы:
Тип X: <class 'pandas.core.series.Series'>
Тип y: <class 'str'>


<p class="task" id="2"></p>

2\. Опишите класс `BankDataset`. Решение должно удовлетворять всем критериям из предыдущего задания, а также:
* при создании объекта в конструктор может быть передан необязательные аргументы `transform` и `target_transform`;
* если аргумент `transform` был передан, то при получении `i`-го элемента, нужно вызвать `transform(x)` и вернуть полученный результат.
* если аргумент `target_transform` был передан, то при получении `i`-го элемента, нужно вызвать `target_transform(y)` и вернуть полученный результат.

Создайте объект класса `BankDataset` и продемонстрируйте работоспособность (без передачи `target_transform` и `transform`).

- [ ] Проверено на семинаре

In [126]:
from typing import Callable, Tuple, Any

class BankDataset(BankDatasetBase):
    def __init__(
        self,
        data: pd.DataFrame,
        transform: Callable | None = None,
        target_transform: Callable | None = None
    ) -> None:
        super().__init__(data) #наследование класса BankDatasetBase
        self.transform = transform
        self.target_transform = target_transform

    def __getitem__(self, idx: int) -> Tuple[Any, Any]:
        x, y = super().__getitem__(idx) #наследуем метод getitem от BankDatasetBase, получаем feature и target

        if self.transform is not None:
            x = self.transform(x)

        if self.target_transform is not None:
            y = self.target_transform(y)

        return x, y

In [127]:
data2 = BankDataset(df)

print(len(data2)) #Длина и тип
print(type(data2))

x, y = data2[0] #Первый элемент
print(x)
print("Метка:", y)

x, y = data2[666] #666-ый элемент
print(x)
print("Метка:", y)

45211
<class '__main__.BankDataset'>
age                  58
job          management
marital         married
education      tertiary
default              no
balance            2143
housing             yes
loan                 no
contact         unknown
day                   5
month               may
duration            261
campaign              1
pdays                -1
previous              0
poutcome        unknown
Name: 0, dtype: object
Метка: no
age                 35
job             admin.
marital         single
education    secondary
default             no
balance            148
housing            yes
loan                no
contact        unknown
day                  6
month              may
duration           128
campaign             2
pdays               -1
previous             0
poutcome       unknown
Name: 666, dtype: object
Метка: no


<p class="task" id="3"></p>

3\. Опишите класс `OrdinalEncoderTransform`. Решение должно удовлетворять следующим критериям:

* при создании объекта в конструктор передаются названия нечисловых столбцов в датасете
* класс реализует интерфейс `Callable` (`__call__`); метод `__call__` имеет один параметр (признаки) и возвращает набор признаков, в котором нечисловые характеристики закодированы целыми числами;
* состояние объекта (индексы для кодирования) обновляется в момент очередного вызова `__call__` (т.е. все данные сразу никогда не передаются никакому методу объекта).
    
Продемонстрируйте работоспособность, создав объект `BankDataset` и передав при создании объект класса `OrdinalEncoderTransform`.

- [ ] Проверено на семинаре

In [128]:
from typing import Dict

In [129]:
class Transform:
    pass

In [130]:
class OrdinalEncoderTransform(Transform):
    def __init__(self, category_columns: list[str]) -> None:
        self.category_columns = category_columns
        self.mappings: Dict[str, Dict[Any, int]] = {
            col: {} for col in category_columns
        } #Обработка категориальных столбцов, для каждого кат. значения свой словарь
        self.next_id: Dict[str, int] = {
            col: 0 for col in category_columns
        } #Следующий свободный индекс для каждой колонки

    def __call__(self, x: pd.Series) -> pd.Series:
        """
        Принимает одну строку (pd.Series),
        кодирует категориальные столбцы,
        при появлении нового значения — присваивает новое целое число
        """

        x = x.copy() #Сохранение исходного столбца

        for col in self.category_columns:
            if col not in x.index:
                continue

            val = x[col]

            if val in self.mappings[col]: #Dict[col: [val: int]] -> по именованию столбца и значению в строке заменяем на соотв. целое число
                x[col] = self.mappings[col][val]
            else:
                code = self.next_id[col] #Если значение ранее не встречалось, добавляем новую запись в Dict[...]
                self.mappings[col][val] = code
                x[col] = code
                self.next_id[col] += 1

        return x

In [131]:
cat_cols = ['job', 'marital', 'education', 'month', 'poutcome', 'housing']

encoder = OrdinalEncoderTransform(category_columns=cat_cols)
data3 = BankDataset(df, transform=encoder)

for i in [0, 1]:
    features, label = data3[i]
    print(f"[{i:2}]  y = {label:6}   "
          f"job={features['job']:2}  "
          f"marital={features['marital']:1}  "
          f"education={features['education']:1}  "
          f"month={features['month']:2}  "
          f"housing={features['housing']:1}")
data3[0], data3[1]

[ 0]  y = no       job= 0  marital=0  education=0  month= 0  housing=0
[ 1]  y = no       job= 1  marital=1  education=1  month= 0  housing=0


((age               58
  job                0
  marital            0
  education          0
  default           no
  balance         2143
  housing            0
  loan              no
  contact      unknown
  day                5
  month              0
  duration         261
  campaign           1
  pdays             -1
  previous           0
  poutcome           0
  Name: 0, dtype: object,
  'no'),
 (age               44
  job                1
  marital            1
  education          1
  default           no
  balance           29
  housing            0
  loan              no
  contact      unknown
  day                5
  month              0
  duration         151
  campaign           1
  pdays             -1
  previous           0
  poutcome           0
  Name: 1, dtype: object,
  'no'))

<p class="task" id="4"></p>

4\. Опишите класс `LabelEncoderTransform`. Решение должно удовлетворять следующим критериям:

* класс реализует интерфейс `Callable` (`__call__`); метод `__call__` имеет один параметр (строку) и возвращает целое число, соответствующее этой строке;
* состояние объекта (индексы для кодирования) обновляется в момент очередного вызова `__call__` (т.е. все данные сразу никогда не передаются никакому методу объекта).
    
Продемонстрируйте работоспособность, создав объект `BankDataset` и передав при создании объекта в качестве аргумента `target_transform` объект класса `LabelEncoderTransform`.

- [ ] Проверено на семинаре

In [132]:
class LabelEncoderTransform(Transform):
    def __init__(self) -> None:
        self.mappings: Dict[str: int] = {} #Dict[str: int] - соответствие строке целому числу
        self.next_id: int = 0

    def __call__(self, x: str) -> int:
      if x not in self.mappings:
        self.mappings[x] = self.next_id
        self.next_id += 1

      return self.mappings[x]

In [133]:
target_encoder = LabelEncoderTransform()

data4 = BankDataset(
    data=df,
    target_transform=target_encoder
)

for i in [0, 1]:
    features, label = data4[i]
    print(f"[{i:2}]  y = {label:2}   "
          f"age = {features['age']:3}   "
          f"job = {features['job']:2}   "
          f"duration = {features['duration']:4}")
data4[0], data4[1]

[ 0]  y =  0   age =  58   job = management   duration =  261
[ 1]  y =  0   age =  44   job = technician   duration =  151


((age                  58
  job          management
  marital         married
  education      tertiary
  default              no
  balance            2143
  housing             yes
  loan                 no
  contact         unknown
  day                   5
  month               may
  duration            261
  campaign              1
  pdays                -1
  previous              0
  poutcome        unknown
  Name: 0, dtype: object,
  0),
 (age                  44
  job          technician
  marital          single
  education     secondary
  default              no
  balance              29
  housing             yes
  loan                 no
  contact         unknown
  day                   5
  month               may
  duration            151
  campaign              1
  pdays                -1
  previous              0
  poutcome        unknown
  Name: 1, dtype: object,
  0))

<p class="task" id="5"></p>

5\. Опишите класс `ToTensor`.  Решение должно удовлетворять следующим критериям:
* класс реализует интерфейс `Callable` (`__call__`); метод `__call__` принимает на вход серию или фрейм и возвращает тензор.

Опишите класс `Compose`.  Решение должно удовлетворять следующим критериям:
* при создании объекта в конструктор передается список объектов `transforms`, каждый из которых имеет метод `__call__(x, y)`;
* класс реализует интерфейс `Callable` (`__call__`); метод `__call__` принимает имеет параметра (признаки и класс в числовом виде) и и возвращает кортеж, полученный путем последовательного вызова объектов из `transforms`.

Продемонстрируйте работоспособность, создав объект `BankDataset` и передав при создании преобразования `Compose` список из объектов LabelEncoderTransform и ToTensor.

- [ ] Проверено на семинаре

In [134]:
import torch as th
from typing import Union, List

class ToTensor(Transform):
    """
    Преобразует pd.Series (признаки) или число (метку) в torch.Tensor,
    Признаки уже числовые
    """
    def __call__(self, x: Union[pd.Series, int, float]) -> th.Tensor:
        if isinstance(x, (pd.Series, pd.DataFrame)): #Доп. проверка на нужный формат данных
            arr = x.values.astype('float32')
            return th.tensor(arr)

        elif isinstance(x, (int, float)):
            return th.tensor(float(x), dtype=th.float32)

        else:
            raise TypeError(f"Unsupported type for ToTensor: {type(x)}")

class Compose(Transform):
    """
    Последовательно применяет список трансформеров,
    Каждый трансформер должен принимать (X, y) и возвращать (X', y').
    """
    def __init__(self, transforms: List[Transform]) -> None:
        self.transforms = transforms

    def __call__(self, x: any):
        for t in self.transforms:
            x = t(x)
        return x

In [135]:
feature_encoder = OrdinalEncoderTransform(
    category_columns=[
        'job', 'marital', 'education', 'default',
        'housing', 'loan', 'contact', 'month', 'poutcome'
    ]
)
label_encoder = LabelEncoderTransform()
to_tensor = ToTensor()

'''
compose = Compose([ #Нужен для поочередного применения транформеров к данным
    feature_encoder, #X -> целое число
    label_encoder, #y → целое число
    to_tensor #Перевод в пару тензор, тензор
])
'''

data5 = BankDataset(
    data=df,
    transform=Compose([feature_encoder, to_tensor]),
    target_transform=Compose([label_encoder, to_tensor])
)

for i in [0, 1]:
    X_tensor, y_tensor = data5[i]
    print(f"[{i:2}]  y = {y_tensor.item():.0f} "
          f"shape(X) = {X_tensor.shape} "
          f"age = {X_tensor[0]:5.1f} "
          f"duration = {X_tensor[10]:6.1f}")
X_tensor, y_tensor

[ 0]  y = 0 shape(X) = torch.Size([16]) age =  58.0 duration =    0.0
[ 1]  y = 0 shape(X) = torch.Size([16]) age =  44.0 duration =    0.0


(tensor([ 44.,   1.,   1.,   1.,   0.,  29.,   0.,   0.,   0.,   5.,   0., 151.,
           1.,  -1.,   0.,   0.]),
 tensor(0.))

<p class="task" id="6"></p>

6\. Разделите датасет из предыдущего задания на обучающую и тестовую выборку в соотношении 75% на 25%. Создайте объект `DataLoader` для получения пакетов размера 64, полученных из перемешанного обучающего датасета. Кастомизируйте `DataLoader` таким образом, чтобы пакет признаков был представлен в виде трехмерного тензора размера 64x2x8 (разделите 16 признаков на два тензора по 8). Получите один пакет и выведите на экран размерность тензоров пакета.

- [ ] Проверено на семинаре

In [136]:
from torch.utils.data import DataLoader, random_split

dataset_size = len(data5)
train_size = int(0.75 * dataset_size)
test_size = dataset_size - train_size

train_data5, test_data5 = random_split(
    data5,
    [train_size, test_size],
    generator=th.Generator().manual_seed(42)   #для воспроизводимости
)

print(f"Размер обучающей выборки:   {len(train_data5):6,d}")
print(f"Размер тестовой выборки:     {len(test_data5):6,d}")
print()


def custom_collate_fn(batch):
    """
    batch — список кортежей (x, y), где
      x.shape == [16]
      y.shape == [] или [1]
    """
    xs = []
    ys = []

    for x, y in batch:
        xs.append(x)
        ys.append(y)

    #Стек по нулевой размерности → [batch_size, 16]
    xs = th.stack(xs)           # shape: [64, 16]
    ys = th.stack(ys)           # shape: [64]

    # Разделяем 16 признаков на две группы по 8
    xs = xs.view(-1, 2, 8)         # [64, 2, 8]

    return xs, ys


#DataLoader только для обучения
train_loader = DataLoader(
    train_data5,
    batch_size=64,
    shuffle=True,
    num_workers=0,               #можно увеличить при необходимости
    drop_last=True,              #чтобы последний батч не был меньше 64
    collate_fn=custom_collate_fn
)

Размер обучающей выборки:   33,908
Размер тестовой выборки:     11,303

